## Summary

### What to do next:

1. **For IMERG**: 
   - Ensure NASA Earthdata credentials are in `~/.netrc`
   - Run from command line: `python scripts/imerg_download.py`
   - Or from notebook: uncomment and run the IMERG cell above

2. **For EarthCARE**:
   - Set up G-Portal account and credentials
   - Update `scripts/config.py` with your credentials
   - Implement G-Portal API calls in `scripts/earthcare_download.py` (template provided)
   - Reference: `GPortalUserManual_en.pdf`

3. **Next steps**:
   - Once data is downloaded, use `satellite_preprocessing.py` to combine and crop files
   - Use `comparison_plotting.py` to generate dropsonde vs satellite comparison figures
   - See SPEC: `notes/SPEC_sonde_vs_satellite.md`

In [1]:
# Uncomment and run to download EarthCARE data
# NOTE: Requires G-Portal account credentials in scripts/config.py
# Currently this is a template - implement G-Portal API calls first

date_range = ("2024-08-10", "2024-09-30")
files = download_earthcare(
    bbox=bbox_ec,
    date_range=date_range,
    output_dir=output_dir_ec,
    product="MSI_L2_PRE",
    skip_existing=True,
)
print(f"\nDownloaded {len(files)} files")


NameError: name 'download_earthcare' is not defined

In [ ]:
from scripts.earthcare_download import download_earthcare
from scripts.config import default_earthcare_bbox, default_earthcare_input_dir, earthcare_credentials

# Show EarthCARE configuration
bbox_ec = default_earthcare_bbox()
output_dir_ec = default_earthcare_input_dir()
creds = earthcare_credentials()

print("EarthCARE Download Configuration:")
print(f"  Bounding box: lat[{bbox_ec.lat_min}, {bbox_ec.lat_max}], lon[{bbox_ec.lon_min}, {bbox_ec.lon_max}]")
print(f"  Output directory: {output_dir_ec}")
print(f"  Date range: 2024-08-10 to 2024-09-30")
print(f"  Product: MSI_L2_PRE (precipitation)")
print(f"\nG-Portal Credentials:")
print(f"  Username: {creds.get('username', 'NOT SET')}")
print(f"  G-Portal URL: {creds.get('gportal_url', 'NOT SET')}")
print(f"\n⚠️  To use EarthCARE, update credentials in scripts/config.py")


## Interactive EarthCARE Download

Below are steps to set up and download EarthCARE data.
The `earthcare_download.py` script is a template - you need to implement the actual API calls.

In [ ]:
# Uncomment and run to download IMERG data
# NOTE: Requires NASA Earthdata account credentials in ~/.netrc
# This may take several hours depending on file size and network speed

# date_range = ("2024-08-10", "2024-09-30")
# files = download_imerg(
#     bbox=bbox,
#     date_range=date_range,
#     output_dir=output_dir,
#     skip_existing=True,
# )
# print(f"\nDownloaded {len(files)} files")


## Interactive IMERG Download

Below are steps to download IMERG data interactively in this notebook.
For production runs, use the command-line script instead.

In [ ]:
import sys
sys.path.insert(0, '/home/565/zr7147/Proj')

# Import the download functions
from scripts.imerg_download import download_imerg
from scripts.config import default_imerg_bbox, default_imerg_input_dir

# Show what will be downloaded
bbox = default_imerg_bbox()
output_dir = default_imerg_input_dir()

print("IMERG Download Configuration:")
print(f"  Bounding box: lat[{bbox.lat_min}, {bbox.lat_max}], lon[{bbox.lon_min}, {bbox.lon_max}]")
print(f"  Output directory: {output_dir}")
print(f"  Date range: 2024-08-10 to 2024-09-30")
print(f"  Product: GPM_3IMERGHH (half-hourly)")


## EarthCARE Download Details

**EarthCARE** (Earth Clouds, Aerosols and Radiation Explorer) is a JAXA mission providing cloud and precipitation measurements.

### Setup (One-time)
1. Create G-Portal account: https://gportal.jaxa.jp/
2. Register for EarthCARE data access
3. Update credentials in `scripts/config.py`:
   ```python
   def earthcare_credentials():
       return {
           "username": "your_username",
           "password": "your_password",
           "gportal_url": "https://gportal.jaxa.jp/",
       }
   ```
   Or set environment variables:
   ```bash
   export EARTHCARE_USERNAME=your_username
   export EARTHCARE_PASSWORD=your_password
   export EARTHCARE_GPORTAL_URL=https://gportal.jaxa.jp/
   ```
4. See: `GPortalUserManual_en.pdf` in project root

### Download Format
- **Product**: MSI_L2_PRE (Level 2 Precipitation, Microphysics)
- **Spatial resolution**: TBD (depends on product level)
- **Format**: NetCDF
- **Download tool**: To be implemented (see `scripts/earthcare_download.py`)

### ORCESTRA Configuration
Default search area (from config.py):
- Latitude: 0°N to 30°N
- Longitude: 70°W to 0°W
- Period: 2024-08-10 to 2024-09-30

### Implementation Note
The `earthcare_download.py` script is a **template**. 
The actual API calls depend on G-Portal's current API. 
Users should:
1. Consult `GPortalUserManual_en.pdf` for current API details
2. Implement the search and download logic in the placeholder section
3. Test with a small bounding box first

## IMERG Download Details

**IMERG** (Integrated Multi-satellitE Retrievals for Global precipitation) is from NASA's GPM (Global Precipitation Measurement) mission.

### Setup (One-time)
1. Create NASA Earthdata account: https://urs.earthdata.nasa.gov
2. Create `~/.netrc` file with credentials:
   ```
   machine urs.earthdata.nasa.gov
   login <username>
   password <password>
   ```
3. Make it readable: `chmod 600 ~/.netrc`

### Download Format
- **Product**: GPM_3IMERGHH (Half-Hourly Final Run)
- **Spatial resolution**: 0.1° x 0.1°
- **Format**: HDF5 (can be read with h5netcdf or netcdf4)
- **Download tool**: earthaccess Python library

### ORCESTRA Configuration
Default search area (from config.py):
- Latitude: 0°N to 30°N
- Longitude: 70°W to 0°W
- Period: 2024-08-10 to 2024-09-30
- Expected: ~2500 half-hourly files

In [ ]:
import earthaccess
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import os

# 1. Authenticate using the ~/.netrc we just created
auth = earthaccess.login()

# 2. Define your Gadi Scratch directory for storage (Home is too small!)
# Replace 'zr7147' with your NCI username if different
output_dir = "/g/data/k10/zr7147/GPM_IMERG_Data"
os.makedirs(output_dir, exist_ok=True)

print(f"Authenticated. Data will be saved to: {output_dir}")

Authenticated. Data will be saved to: /g/data/k10/zr7147/GPM_IMERG_Data


In [ ]:
# Define ORCESTRA region: [Lon Min, Lat Min, Lon Max, Lat Max]
bbox = (-65, 5, -15, 20) 

# The campaign Date Range (YYYY-MM-DD)
date_range = ("2024-08-10", "2024-09-30")

# Search for GPM IMERG Final Run (Half-Hourly)
results = earthaccess.search_data(
    short_name="GPM_3IMERGHH",
    bounding_box=bbox,
    temporal=date_range
)

print(f"Found {len(results)} half-hourly files for these dates.")

# Download only the files we don't already have
downloaded_files = earthaccess.download(results, local_path=output_dir)

Found 2496 half-hourly files for these dates.


QUEUEING TASKS | :   0%|          | 0/2496 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/2496 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/2496 [00:00<?, ?it/s]

In [ ]:
# Load all downloaded files into one dataset
# IMERG files usually have names like '3B-HHR.MS.MRG.3IMERG...'
file_pattern = os.path.join(output_dir, "*.HDF5") 
ds_gpm = xr.open_mfdataset(file_pattern, concat_dim='time', combine='nested', engine='h5netcdf', group='/Grid')

# Extract 'precipitation' (Calibrated Precipitation in mm/hr)
# Note: IMERG longitude is -180 to 180. We slice to our flight region.
precip = ds_gpm['precipitation'].sel(lon=slice(-65, -15), lat=slice(5, 20))

# Calculate a daily mean for the overview plot
daily_mean = precip.mean(dim='time')

/scratch/nf33/zr7147/tmp/ipykernel_2092433/1558644469.py:4: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds_gpm = xr.open_mfdataset(file_pattern, concat_dim='time', combine='nested', engine='h5netcdf', group='/Grid')
